# CURE-Rec — reviewer-revision experiments

This notebook contains **new evidence**, not a rerun of accepted results. Every expensive action is disabled by default. It addresses held-out portfolio evaluation and selector baselines without inventing values.


## Setup


In [3]:
from pathlib import Path
import importlib
import sys

CWD = Path.cwd().resolve()
CANDIDATES = [CWD, CWD / 'paper-ideas' / 'CURE-Rec' / 'code', *CWD.parents]
ROOT = next((p for p in CANDIDATES if (p / 'pyproject.toml').exists() and (p / 'cure_rec').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the CURE-Rec code directory or repository root.')
sys.path[:] = [str(ROOT), *[item for item in sys.path if item != str(ROOT)]]
for name in list(sys.modules):
    if name == 'cure_rec' or name.startswith('cure_rec.'):
        del sys.modules[name]
importlib.invalidate_caches()

from cure_rec.config import load_settings
from cure_rec.revision import recompute_selector_summary, run_selector_holdout_study

FULL_CONFIG = ROOT / 'configs' / 'curesim_full.yaml'
RUN_ROOT = ROOT / 'runs'
print('CURE-Rec source:', ROOT)


CURE-Rec source: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code


## Action 1 — held-out portfolio-selection benchmark

This is the reviewer-required experiment. Portfolios are selected on `SELECTION_SEEDS`, frozen, then evaluated on disjoint `EVALUATION_SEEDS`. It compares exact CURE maximin against singleton, robust-Shapley, greedy, nominal-only, random-feasible, and grand-coalition diagnostic selectors.

**Cost:** every selection/evaluation seed executes an exact 64-coalition game. Start with the smoke configuration, inspect the files, then deliberately enable the full configuration.


In [3]:
RUN_SELECTOR_HOLDOUT_SMOKE = False
RUN_SELECTOR_HOLDOUT_FULL = True

assert not (RUN_SELECTOR_HOLDOUT_SMOKE and RUN_SELECTOR_HOLDOUT_FULL)

if RUN_SELECTOR_HOLDOUT_SMOKE:
    cfg = load_settings(ROOT / 'configs' / 'curesim_quickstart.yaml')
    cfg.run.output_root = RUN_ROOT
    revision_run = run_selector_holdout_study(
        cfg, selection_seeds=(42,), evaluation_seeds=(200, 201)
    )
    print('Selector holdout smoke run:', revision_run)
elif RUN_SELECTOR_HOLDOUT_FULL:
    cfg = load_settings(FULL_CONFIG)
    cfg.run.output_root = RUN_ROOT
    revision_run = run_selector_holdout_study(
        cfg, selection_seeds=(42, 43, 44, 45, 46), evaluation_seeds=tuple(range(200, 220))
    )
    print('Selector holdout full run:', revision_run)
else:
    print('Selector holdout study disabled. Enable smoke first; full run is expensive.')


2026-08-08 20:34:20,917 | INFO | run_started | {"config_hash": "bf4e977926ddc637", "run_id": "selector-selection-42-20260808T193420Z-dedeaefc"}
2026-08-08 20:34:20,918 | INFO | exact_game_started | {}
2026-08-08 20:34:20,918 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "nominal"}
2026-08-08 20:36:57,594 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.13710546477148466, "scenario": "nominal", "shapley_efficiency_gap": 0.0}
2026-08-08 20:36:57,594 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "fatigue_stress"}
2026-08-08 20:39:32,554 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.13194331906701645, "scenario": "fatigue_stress", "shapley_efficiency_gap": 2.7755575615628914e-17}
2026-08-08 20:39:32,555 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "popularity_stress"}
2026-08-08 20:42:07,351 | INFO | scenario_game_completed

## Action 2 — inspect completed revision output

Set the exact run directory printed by Action 1. This is cheap. The summary contains held-out robust utility, feasibility, paired differences versus exact CURE maximin, standardized paired effect, and an exact sign-test p-value.


In [6]:
import pandas as pd
REVISION_OUTPUT = (
    RUN_ROOT / "reviewer-selector-holdout-20260810T012332Z"
)
if REVISION_OUTPUT is not None:
    # Stand-alone import makes this cell safe after a pull even when an earlier
    # kernel setup cell imported the pre-correction revision module.
    from importlib import reload
    import cure_rec.revision as revision
    revision = reload(revision)
    REVISION_OUTPUT = Path(REVISION_OUTPUT)
    display(pd.read_csv(REVISION_OUTPUT / 'selection_choices.csv'))
    display(revision.recompute_selector_summary(REVISION_OUTPUT))
    display(pd.read_csv(REVISION_OUTPUT / 'heldout_selector_evaluations.csv').head())
else:
    print('Set REVISION_OUTPUT to inspect a completed selector study.')


,selector,selected_mask,selected_interventions,robust_objective,feasible,selection_seed
0,cure_exact_maximin,1,repeat_cap,0.298514,True,42
1,best_singleton,1,repeat_cap,0.298514,True,42
2,robust_shapley_1,1,repeat_cap,0.298514,True,42
3,robust_shapley_budget,1,repeat_cap,0.298514,True,42
4,greedy_robust,1,repeat_cap,0.298514,True,42
5,nominal_scenario,1,repeat_cap,0.298514,True,42
6,random_feasible,3,repeat_cap;explore_slot,0.198068,True,42
7,grand_coalition_diagnostic,0,NaN,0.000000,False,42
8,cure_exact_maximin,1,repeat_cap,0.296202,True,43
9,best_singleton,1,repeat_cap,0.296202,True,43


,selector,selection_evaluation_pairs,independent_evaluation_seeds,robust_lower_improvement_mean,robust_lower_improvement_std,feasible_rate,paired_difference_vs_cure_mean,paired_difference_vs_cure_sd,paired_effect_dz,exact_sign_test_p
0,best_singleton,30,10,0.294735,0.002385,1.0,0.000000,0.000000,NaN,1.000000
1,cure_exact_maximin,30,10,0.294735,0.002385,1.0,0.000000,0.000000,NaN,1.000000
2,grand_coalition_diagnostic,30,10,0.000000,0.000000,0.7,-0.294735,0.002472,-119.222726,0.001953
3,greedy_robust,30,10,0.294735,0.002385,1.0,0.000000,0.000000,NaN,1.000000
4,nominal_scenario,30,10,0.294735,0.002385,1.0,0.000000,0.000000,NaN,1.000000
5,random_feasible,30,10,0.005162,0.183545,1.0,-0.289573,0.000945,-306.321592,0.001953
6,robust_shapley_1,30,10,0.294735,0.002385,1.0,0.000000,0.000000,NaN,1.000000
7,robust_shapley_budget,30,10,0.294735,0.002385,1.0,0.000000,0.000000,NaN,1.000000


,selection_seed,evaluation_seed,selector,selected_mask,selected_interventions,robust_lower_improvement,scenario_lower_improvement,scenario_upper_improvement,feasible,cost,relevance_delta_lower,provider_disparity_upper,fatigue_upper
0,42,200,cure_exact_maximin,1,repeat_cap,0.29664,0.29664,0.311755,True,0.05,-0.035794,0.217662,0.0
1,42,200,best_singleton,1,repeat_cap,0.29664,0.29664,0.311755,True,0.05,-0.035794,0.217662,0.0
2,42,200,robust_shapley_1,1,repeat_cap,0.29664,0.29664,0.311755,True,0.05,-0.035794,0.217662,0.0
3,42,200,robust_shapley_budget,1,repeat_cap,0.29664,0.29664,0.311755,True,0.05,-0.035794,0.217662,0.0
4,42,200,greedy_robust,1,repeat_cap,0.29664,0.29664,0.311755,True,0.05,-0.035794,0.217662,0.0


## Required interpretation

The held-out selector study is simulator-conditional evidence. It does not turn MovieLens into causal data. Do not replace the current manuscript values with these results until the full run, archive, and manuscript tables have been reviewed.


In [1]:
!cure-rec revision-ablation \
    --config ../configs/curesim_full.yaml \
    --output-dir ../results/reviewer_phase_assets

2026-08-13 14:23:18,972 | INFO | run_started | {"config_hash": "131553b36a3735d7", "run_id": "curesim-full-20260813T132318Z-e88f0d33"}
2026-08-13 14:23:18,973 | INFO | exact_game_started | {}
2026-08-13 14:23:18,974 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "nominal"}
2026-08-13 14:27:40,078 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.13323832372399413, "scenario": "nominal", "shapley_efficiency_gap": 1.3877787807814457e-16}
2026-08-13 14:27:40,079 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "fatigue_stress"}
2026-08-13 14:32:05,070 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.13544545518030754, "scenario": "fatigue_stress", "shapley_efficiency_gap": 0.0}
2026-08-13 14:32:05,071 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "popularity_stress"}
2026-08-13 14:35:30,594 | INFO | scenario_game_completed | {"gran

In [2]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), Path.cwd()/"paper-ideas/CURE-Rec/code", *Path.cwd().parents] if (p/"cure_rec").exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from cure_rec.revision_suite import crn_paired_difference, summarize_crn, paired_user_statistics
from cure_rec.reviewer_cli import run_holdout, aggregate

CONFIG = ROOT / "configs/curesim_full.yaml"
OUTPUT = ROOT.parent / "results" / "reviewer_phase_assets"
RUN_PHASE_A = False
RUN_PHASE_BC = False  # use the CLI command below; this cell is intentionally non-rerunning
RUN_CRN = True
RUN_EXTERNAL_STATS = False

if RUN_PHASE_A:
    holdout = run_holdout(CONFIG, ROOT / "runs" / "reviewer-revision", (42,43,44,45,46), tuple(range(200,220)))
    print(holdout)

# Phase B/C command (run from the repository root):
# cure-rec revision-ablation --config configs/curesim_full.yaml --output-dir ../results/reviewer_phase_assets

if RUN_CRN:
    # Replace these deterministic fixtures with the simulator's fixed coalition
    # utility samplers for the final CRN experiment.
    base = lambda seed: 0.1 + 0.01 * ((seed * 17) % 11)
    treated = lambda seed: 0.2 + 0.01 * ((seed * 17 + 3) % 11)
    crn = crn_paired_difference(base, treated, range(300,320), independent=False)
    iid = crn_paired_difference(base, treated, range(300,320), independent=True)
    print(summarize_crn(crn, iid))

if RUN_EXTERNAL_STATS:
    # `per_user_metrics.csv` must contain user_id, model, hit, and ndcg.
    metrics = __import__('pandas').read_csv(OUTPUT / 'per_user_metrics.csv')
    ci, tests = paired_user_statistics(metrics)
    ci.to_csv(OUTPUT / 'paired_bootstrap_ci.csv', index=False)
    tests.to_csv(OUTPUT / 'paired_tests_holm.csv', index=False)
    display(ci); display(tests)


{'crn_variance': 0.002674736842105263, 'independent_variance': 0.003152368421052632, 'variance_ratio': 0.8484848484848484, 'crn_n': 20, 'independent_n': 20}
